In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [ ]:
news_dataset = pd.read_csv(
    "train.csv",
    engine="python",
    on_bad_lines="skip"
)

In [ ]:
news_dataset.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [ ]:
news_dataset.shape

(20822, 5)

In [ ]:
news_dataset.dtypes

,0
id,object
title,object
author,object
text,object
label,object


In [ ]:
news_dataset["id"] = pd.to_numeric(news_dataset["id"], errors="coerce")
news_dataset["label"] = pd.to_numeric(news_dataset["label"], errors="coerce")

In [ ]:
news_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20822 entries, 0 to 20821
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      20798 non-null  float64
 1   title   20257 non-null  object 
 2   author  18847 non-null  object 
 3   text    20763 non-null  object 
 4   label   20798 non-null  float64
dtypes: float64(2), object(3)
memory usage: 813.5+ KB


In [ ]:
news_dataset.dtypes

,0
id,float64
title,object
author,object
text,object
label,float64


In [ ]:
news_dataset.describe()

,id,label
count,20798.000000,20798.000000
mean,10398.899077,0.500577
std,6004.485395,0.500012
min,0.000000,0.000000
25%,5199.250000,0.000000
50%,10398.500000,1.000000
75%,15598.750000,1.000000
max,20799.000000,1.000000


In [ ]:
news_dataset.isnull().sum()

,0
id,24
title,565
author,1975
text,59
label,24


In [ ]:
news_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20822 entries, 0 to 20821
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      20798 non-null  float64
 1   title   20257 non-null  object 
 2   author  18847 non-null  object 
 3   text    20763 non-null  object 
 4   label   20798 non-null  float64
dtypes: float64(2), object(3)
memory usage: 813.5+ KB


In [ ]:
news_dataset["title"] = news_dataset["title"].fillna("")
news_dataset["author"] = news_dataset["author"].fillna("")
news_dataset["text"] = news_dataset["text"].fillna("")

In [ ]:
news_dataset["id"] = news_dataset["id"].fillna(0).astype(int)
news_dataset["label"] = news_dataset["label"].fillna(0).astype(int)

In [ ]:
news_dataset.isnull().sum()

,0
id,0
title,0
author,0
text,0
label,0


In [ ]:
news_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20822 entries, 0 to 20821
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      20822 non-null  int64 
 1   title   20822 non-null  object
 2   author  20822 non-null  object
 3   text    20822 non-null  object
 4   label   20822 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 813.5+ KB


In [ ]:
news_dataset["content"] = news_dataset["author"] + " " + news_dataset["title"]

In [ ]:
print(news_dataset["content"])

0        Darrell Lucus House Dem Aide: We Didn’t Even S...
1        Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo...
2        Consortiumnews.com Why the Truth Might Get You...
3        Jessica Purkiss 15 Civilians Killed In Single ...
4        Howard Portnoy Iranian woman jailed for fictio...
                               ...                        
20817    Jerome Hudson Rapper T.I.: Trump a ’Poster Chi...
20818    Benjamin Hoffman N.F.L. Playoffs: Schedule, Ma...
20819    Michael J. de la Merced and Rachel Abrams Macy...
20820    Alex Ansary NATO, Russia To Hold Parallel Exer...
20821              David Swanson What Keeps the F-35 Alive
Name: content, Length: 20822, dtype: object


In [ ]:
news_dataset.head()

,id,title,author,text,label,content
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1,Darrell Lucus House Dem Aide: We Didn’t Even S...
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0,"Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo..."
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1,Consortiumnews.com Why the Truth Might Get You...
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1,Jessica Purkiss 15 Civilians Killed In Single ...
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1,Howard Portnoy Iranian woman jailed for fictio...


In [ ]:
news_dataset.shape

(20822, 6)

In [ ]:
X = news_dataset.drop(columns="label", axis=1)
y = news_dataset["label"]

In [ ]:
X

,id,title,author,text,content
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus House Dem Aide: We Didn’t Even S...
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,"Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo..."
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",Consortiumnews.com Why the Truth Might Get You...
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,Jessica Purkiss 15 Civilians Killed In Single ...
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,Howard Portnoy Iranian woman jailed for fictio...
...,...,...,...,...,...
20817,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...,Jerome Hudson Rapper T.I.: Trump a ’Poster Chi...
20818,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...,"Benjamin Hoffman N.F.L. Playoffs: Schedule, Ma..."
20819,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...,Michael J. de la Merced and Rachel Abrams Macy...
20820,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal...","Alex Ansary NATO, Russia To Hold Parallel Exer..."


In [ ]:
y

,label
0,1
1,0
2,1
3,1
4,1
...,...
20817,0
20818,0
20819,0
20820,1


In [ ]:
stop_words = set(stopwords.words("english"))

stem_porter = PorterStemmer()

def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]', ' ', str(content))
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [stem_porter.stem(word) for word in stemmed_content if not word in stop_words]
  stemmed_content = ' '.join(stemmed_content)
  return stemmed_content

In [ ]:
news_dataset["content"] = news_dataset["content"].apply(stemming)

In [ ]:
news_dataset["content"]

,content
0,darrel lucu hous dem aid even see comey letter...
1,daniel j flynn flynn hillari clinton big woman...
2,consortiumnew com truth might get fire
3,jessica purkiss civilian kill singl us airstri...
4,howard portnoy iranian woman jail fiction unpu...
...,...
20817,jerom hudson rapper trump poster child white s...
20818,benjamin hoffman n f l playoff schedul matchup...
20819,michael j de la merc rachel abram maci said re...
20820,alex ansari nato russia hold parallel exercis ...


In [ ]:
news_dataset["content"].values

array(['darrel lucu hous dem aid even see comey letter jason chaffetz tweet',
       'daniel j flynn flynn hillari clinton big woman campu breitbart',
       'consortiumnew com truth might get fire', ...,
       'michael j de la merc rachel abram maci said receiv takeov approach hudson bay new york time',
       'alex ansari nato russia hold parallel exercis balkan',
       'david swanson keep f aliv'], dtype=object)

In [ ]:
news_dataset["label"]

,label
0,1
1,0
2,1
3,1
4,1
...,...
20817,0
20818,0
20819,0
20820,1


In [ ]:
news_dataset["label"].values

array([1, 0, 1, ..., 0, 1, 1])

In [ ]:
X = news_dataset["content"].values
y = news_dataset["label"].values

In [ ]:
X

array(['darrel lucu hous dem aid even see comey letter jason chaffetz tweet',
       'daniel j flynn flynn hillari clinton big woman campu breitbart',
       'consortiumnew com truth might get fire', ...,
       'michael j de la merc rachel abram maci said receiv takeov approach hudson bay new york time',
       'alex ansari nato russia hold parallel exercis balkan',
       'david swanson keep f aliv'], dtype=object)

In [ ]:
y

array([1, 0, 1, ..., 0, 1, 1])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

In [ ]:
print(X.shape, X_train.shape, X_test.shape)

(20822,) (16657,) (4165,)


In [ ]:
vectorizer = TfidfVectorizer()

In [ ]:
X_train = vectorizer.fit_transform(X_train)

X_test = vectorizer.transform(X_test)

In [ ]:
model = LogisticRegression()

In [ ]:
model.fit(X_train, y_train)

LogisticRegression()

In [ ]:
training_data_prediction = model.predict(X_train)

In [ ]:
training_data_accuracy = accuracy_score(training_data_prediction, y_train)

In [ ]:
print("Accuracy score of the training data: ", training_data_accuracy)

Accuracy score of the training data:  0.9864921654559644


In [ ]:
testing_data_prediction = model.predict(X_test)

In [ ]:
testing_data_accuracy = accuracy_score(testing_data_prediction, y_test)

In [ ]:
print("Accuracy score of the testing data: ", testing_data_accuracy)

Accuracy score of the testing data:  0.9747899159663865


In [ ]:
X_news = X_test[1]

prediction = model.predict(X_news)
print(prediction)

if prediction[0] == 0:
  print("The news is Real")
else:
  print("The news is Fake")

[0]
The news is Real


In [ ]:
print(y_test[1])

0
